In [1]:
# this is the workbook where I work on the decline curve analysis

In [1]:
#basics
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# plotly and dash
import dash
from dash import dcc, html
import plotly.offline as pyo
import plotly.graph_objs as go
from dash.dependencies import Input, Output
from dash.exceptions import PreventUpdate
import plotly.express as px

#others
from pathlib import Path
import re

In [2]:
import sys
sys.path.append("../src")   # VERY IMPORTANT

## Loading and Processing production data

In [14]:
from Load_clean_production_data import Load_clean_production_data
file_path = ("../../0_Volve_dataset/5_Production_data/Volve_production_data.xlsx")
production_df = Load_clean_production_data(file_path)
from Load_clean_production_data import metric_to_field
production_df_field=pd.DataFrame()
production_df_field=metric_to_field(production_df)

In [15]:
production_df_field.columns

Index(['DATEPRD', 'WELL_BORE_CODE', 'NPD_WELL_BORE_CODE', 'NPD_WELL_BORE_NAME',
       'NPD_FIELD_CODE', 'NPD_FIELD_NAME', 'NPD_FACILITY_CODE',
       'NPD_FACILITY_NAME', 'ON_STREAM_HRS', 'AVG_DOWNHOLE_PRESSURE_PSI',
       'AVG_DOWNHOLE_TEMPERATURE_F', 'AVG_DP_TUBING_PSI',
       'AVG_ANNULUS_PRESS_PSI', 'AVG_CHOKE_SIZE_P', 'AVG_CHOKE_UOM',
       'AVG_WHP_P_PSI', 'AVG_WHT_P', 'DP_CHOKE_SIZE', 'BORE_OIL_VOL_STB',
       'BORE_GAS_VOL_MSCF', 'BORE_WAT_VOL_STB', 'BORE_WI_VOL', 'FLOW_KIND',
       'WELL_TYPE'],
      dtype='str')

In [16]:
# drop columns u dont need
production_df_field = production_df_field.drop(columns=[
    col for col in [
        "WELL_BORE_CODE","NPD_WELL_BORE_CODE", "NPD_FIELD_CODE","ON_STREAM_HRS","NPD_FIELD_NAME","NPD_FACILITY_CODE","NPD_FACILITY_NAME","NPD_FIELD_CODE",'BORE_WI_VOL', 'FLOW_KIND',
        'WELL_TYPE', 'AVG_DOWNHOLE_PRESSURE_PSI','AVG_DOWNHOLE_TEMPERATURE_F', 'AVG_DP_TUBING_PSI','AVG_ANNULUS_PRESS_PSI','AVG_WHP_P_PSI', 'AVG_WHT_P'
        ] if col in production_df_field.columns
    ])

In [17]:
# calculate running time and normalize time
from Load_clean_production_data import times_calc
production_df_field=times_calc(production_df_field)


In [18]:
production_df_field

,DATEPRD,NPD_WELL_BORE_NAME,AVG_CHOKE_SIZE_P,AVG_CHOKE_UOM,DP_CHOKE_SIZE,BORE_OIL_VOL_STB,BORE_GAS_VOL_MSCF,BORE_WAT_VOL_STB,Time_zero,Normalized_Time_days,is_producing,Days_on_prod
0,2014-04-07,15/9-F-1 C,0.000000,%,0.00000,0.0,0.0,0.0,2014-04-07,0.0,False,0
1,2014-04-08,15/9-F-1 C,1.003059,%,0.00000,0.0,0.0,0.0,2014-04-07,1.0,False,0
2,2014-04-09,15/9-F-1 C,0.979008,%,0.00000,0.0,0.0,0.0,2014-04-07,2.0,False,0
3,2014-04-10,15/9-F-1 C,0.545759,%,0.00000,0.0,0.0,0.0,2014-04-07,3.0,False,0
4,2014-04-11,15/9-F-1 C,1.215987,%,33.07195,0.0,0.0,0.0,2014-04-07,4.0,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...
15629,2016-09-14,15/9-F-5,0.636088,%,0.01862,0.0,0.0,0.0,2007-09-01,3301.0,False,129
15630,2016-09-15,15/9-F-5,0.670794,%,0.00631,0.0,0.0,0.0,2007-09-01,3302.0,False,129
15631,2016-09-16,15/9-F-5,0.664393,%,0.01181,0.0,0.0,0.0,2007-09-01,3303.0,False,129
15632,2016-09-17,15/9-F-5,0.624660,%,0.02576,0.0,0.0,0.0,2007-09-01,3304.0,False,129


## Decline curve analysis

In [19]:
from lmfit import Parameters, minimize, report_fit

### Helper functions

In [102]:
### Helper function
def get_arps_initial_guess(t, q, n_early=90):
    """
    This is an helper function to get initial guesses for the decline curve analysis
    t is time (days on), array
    q  production, for this dataset stb/day or mscf/day, array
    n_early: the initial amount of days to use to calculate the initial guess for the Di, integer
    returns inital guesses for qi, Di(1/day), and b
    """
    t = np.asarray(t)
    q = np.asarray(q)

    mask = q > 0
    t = t[mask]
    q = q[mask]

    order = np.argsort(t)
    t = t[order]
    q = q[order]

    n = min(n_early, len(q))

    qi_guess = np.percentile(q[:n], 90)

    try:
        slope = np.polyfit(t[:n], np.log(q[:n]), 1)[0]   #Fits a straight line
        Di_guess = max(-slope, 1e-6)
    except:
        Di_guess = 0.001

    b_guess = 0.5

    return [qi_guess, Di_guess, b_guess]

In [103]:
def sort_remove_nan(df, rate):
    """
    This is an helper function to sort the dataframe and make sure no value less then zero is selected
    df: dataframe having time as in days_on prod and rate. rate could be  BORE_OIL_VOL_STB	BORE_GAS_VOL_MSCF	BORE_WAT_VOL_STB
    returns the cleaned dataframe
    """
    df = df.sort_values("Days_on_prod")
    df = df[df[rate] > 0]
    return df

### Exponential decline

In [104]:
def Exponential(t, qi, Di):
    """
    Exponential decline curve analysis
    t is time (days on), array
    qi initial production, array
    Di initial decline rate (per unit time this case 1/day), float
    returns qmodel at every time step, array
    """

    return  qi*np.exp(-Di*t)

In [105]:
def Exponential_residuals_log_lmfit(params, t, q):
    """
    Loss function. returns the difference between the actual data and the model
    params: object like array to take parameters to be optimized
    t is time (days on), array
    q production, array
    """
    qi = params["qi"].value
    Di = params["Di"].value
    
    q_model = Exponential(t, qi, Di)
    
    q_safe = np.maximum(q, 1e-6)
    q_model_safe = np.maximum(q_model, 1e-6)
    
    return np.log(q_model_safe) - np.log(q_safe)

In [106]:
def Exponential_minimizer(t,q):
    """
    This function calls the lmfit optimizer. 
    t: time arrays in days
    q: rate array in stb/day or mscf/day
    returns the fitted qi and Di
    """
    t = np.asarray(t, dtype=float)
    q = np.asarray(q, dtype=float)
    
    params_exponential = Parameters()      # parameters for the exponential distribution
    
    x0 = get_arps_initial_guess(t, q)    # get intial guesses
    
    params_exponential.add("qi", value=x0[0], min=0)
    params_exponential.add("Di", value=x0[1], min=0, max=1)
    
    result = minimize(
        Exponential_residuals_log_lmfit,
        params_exponential,
        args=(t, q),
        method="least_squares",
        loss="soft_l1"
        )
    
    report_fit(result)
    
    qi_fit = result.params["qi"].value
    Di_fit = result.params["Di"].value
    return(qi_fit, Di_fit)


### Hyperbolic Decline

In [107]:
def Hyperbolic(t, qi, Di,b):
    """
    hyperbolic decline curve analysis
    t is time (days on), array
    qi initial production, array
    Di initial decline rate (per unit time this case 1/day), float
    b decline curve exponent
    returns qmodel at every time step, array
    """
    t = np.asarray(t)
    

    return  qi/(1+b*Di*t)**(1/b)

In [108]:
def Hyperbolic_residuals_log_lmfit(params, t, q):
    """
    Loss function. returns the difference between the actual data and the model
    params: object like array to take parameters to be optimized
    t is time (days on), array
    q production, array
    """
    qi = params["qi"].value
    Di = params["Di"].value
    b = params["b"].value
    
    q_model = Hyperbolic(t, qi, Di,b)
    
    q_safe = np.maximum(q, 1e-6)
    q_model_safe = np.maximum(q_model, 1e-6)
    
    return np.log(q_model_safe) - np.log(q_safe)

In [109]:
def Hyperbolic_minimizer(t,q):
    """
    This function calls the lmfit optimizer. 
    t: time arrays in days
    q: rate array in stb/day or mscf/day
    returns the fitted qi and Di and b
    """
    t = np.asarray(t, dtype=float)
    q = np.asarray(q, dtype=float)
    
    params_Hyperbolic = Parameters()      # parameters for the hyperbolic distribution
    
    x0 = get_arps_initial_guess(t, q)    # get intial guesses
    
    params_Hyperbolic.add("qi", value=x0[0], min=0)
    params_Hyperbolic.add("Di", value=x0[1], min=0, max=1)
    params_Hyperbolic.add("b",  value=x0[2], min=0.1, max=0.9999999)
    
    result = minimize(
        Hyperbolic_residuals_log_lmfit,
        params_Hyperbolic,
        args=(t, q),
        method="least_squares",
        loss="soft_l1"
        )
    
    report_fit(result)
    
    qi_fit = result.params["qi"].value
    Di_fit = result.params["Di"].value
    b_fit = result.params["b"].value
    print(qi_fit, Di_fit,b_fit)
    return(qi_fit, Di_fit,b_fit)

### Harmonic Decline

In [110]:
def Harmonic(t, qi, Di,b=1):
    """
    Harmonic decline curve analysis
    t is time (days on), array
    qi initial production, array
    Di initial decline rate (per unit time this case 1/day), float
    returns qmodel at every time step, array
    """

    return  qi/(1+b*Di*t)**(1/b)

In [111]:
def Harmonic_residuals_log_lmfit(params, t, q):
    """
    Loss function. returns the difference between the actual data and the model
    params: object like array to take parameters to be optimized
    t is time (days on), array
    q production, array
    """
    qi = params["qi"].value
    Di = params["Di"].value
    
    q_model = Harmonic(t, qi, Di)
    
    q_safe = np.maximum(q, 1e-6)
    q_model_safe = np.maximum(q_model, 1e-6)
    
    return np.log(q_model_safe) - np.log(q_safe)

In [112]:
def Harmonic_minimizer(t,q):
    """
    This function calls the lmfit optimizer. 
    t: time arrays in days
    q: rate array in stb/day or mscf/day
    returns the fitted qi and Di and b
    """
    
    params_Harmonic = Parameters()      # parameters for the exponential distribution
    
    x0 = get_arps_initial_guess(t, q)    # get intial guesses
    
    params_Harmonic.add("qi", value=x0[0], min=0)
    params_Harmonic.add("Di", value=x0[1], min=0, max=1)
    
    result = minimize(
        Harmonic_residuals_log_lmfit,
        params_Harmonic,
        args=(t, q),
        method="least_squares",
        loss="soft_l1"
        )
    
    report_fit(result)
    
    qi_fit = result.params["qi"].value
    Di_fit = result.params["Di"].value
    return(qi_fit, Di_fit)

In [113]:
production_df_field["NPD_WELL_BORE_NAME"].unique()

<StringArray>
[ '15/9-F-1 C',   '15/9-F-11',   '15/9-F-12',   '15/9-F-14', '15/9-F-15 D',
    '15/9-F-4',    '15/9-F-5']
Length: 7, dtype: str

### Putting everything together

In [144]:
production_df_field

,DATEPRD,NPD_WELL_BORE_NAME,AVG_CHOKE_SIZE_P,AVG_CHOKE_UOM,DP_CHOKE_SIZE,BORE_OIL_VOL_STB,BORE_GAS_VOL_MSCF,BORE_WAT_VOL_STB,Time_zero,Normalized_Time_days,is_producing,Days_on_prod
0,2014-04-07,15/9-F-1 C,0.000000,%,0.00000,0.0,0.0,0.0,2014-04-07,0.0,False,0
1,2014-04-08,15/9-F-1 C,1.003059,%,0.00000,0.0,0.0,0.0,2014-04-07,1.0,False,0
2,2014-04-09,15/9-F-1 C,0.979008,%,0.00000,0.0,0.0,0.0,2014-04-07,2.0,False,0
3,2014-04-10,15/9-F-1 C,0.545759,%,0.00000,0.0,0.0,0.0,2014-04-07,3.0,False,0
4,2014-04-11,15/9-F-1 C,1.215987,%,33.07195,0.0,0.0,0.0,2014-04-07,4.0,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...
15629,2016-09-14,15/9-F-5,0.636088,%,0.01862,0.0,0.0,0.0,2007-09-01,3301.0,False,129
15630,2016-09-15,15/9-F-5,0.670794,%,0.00631,0.0,0.0,0.0,2007-09-01,3302.0,False,129
15631,2016-09-16,15/9-F-5,0.664393,%,0.01181,0.0,0.0,0.0,2007-09-01,3303.0,False,129
15632,2016-09-17,15/9-F-5,0.624660,%,0.02576,0.0,0.0,0.0,2007-09-01,3304.0,False,129


In [165]:
channel="BORE_GAS_VOL_MSCF"
wellname='15/9-F-14'
df=production_df_field[production_df_field['NPD_WELL_BORE_NAME']==wellname]
df=sort_remove_nan(df, channel)

t=df["Days_on_prod"]
q=df[channel]

In [166]:
# time function for predictions

total_predict_days=5475 # 15 years
last_date = df["DATEPRD"].max()
last_days_on = df["Days_on_prod"].max()

calendar_days = (df["DATEPRD"].max()-df["DATEPRD"].min()).days # days on production including when the well is down
calendar_days = max(calendar_days, 1)
# forecast producing days
uptime_fraction = last_days_on / calendar_days
remaining_calendar_days = max(total_predict_days - calendar_days, 0)
t_forecast = np.arange(last_days_on,last_days_on + remaining_calendar_days * uptime_fraction,30)  # I want to start the forecast after (5475-normalized_days) 5475 is 15 years

# estimate uptime
uptime_fraction = (last_days_on /calendar_days)

# convert to calendar days
future_calendar_days = (t_forecast - last_days_on) / uptime_fraction

# forecast dates
forecast_dates = (last_date+pd.to_timedelta(future_calendar_days, unit="D"))

In [167]:
# exponential
qi_exp,Di_exp=Exponential_minimizer(t,q)

q_model_exp= qi_exp*np.exp(-Di_exp*t)
# forecast rates
q_forecast_exp = qi_exp*np.exp(-Di_exp*t_forecast)

[[Fit Statistics]]
    # fitting method   = least_squares
    # function evals   = 20
    # data points      = 2724
    # variables        = 2
    chi-square         = 687.471299
    reduced chi-square = 0.25256109
    Akaike info crit   = -3746.50286
    Bayesian info crit = -3734.68315
[[Variables]]
    qi:  26050.0626 +/- 535.212558 (2.05%) (init = 26049.78)
    Di:  0.00133264 +/- 1.2849e-05 (0.96%) (init = 1e-06)
[[Correlations]] (unreported correlations are < 0.100)
    C(qi, Di) = +0.8648


In [168]:
# Hyperbolic
qi_hyber,Di_hyper, b_hyper=Hyperbolic_minimizer(t,q)

q_model_hyper= qi_hyber/(1+b_hyper*Di_hyper*t)**(1/b_hyper)
# forecast rates
q_forecast_hyper = qi_hyber/(1+b_hyper*Di_hyper*t_forecast)**(1/b_hyper)

[[Fit Statistics]]
    # fitting method   = least_squares
    # function evals   = 48
    # data points      = 2724
    # variables        = 3
    chi-square         = 726.863929
    reduced chi-square = 0.26713118
    Akaike info crit   = -3592.72373
    Bayesian info crit = -3574.99416
[[Variables]]
    qi:  30995.4928 +/- 1111.82384 (3.59%) (init = 26049.78)
    Di:  0.00156327 +/- 7.3834e-05 (4.72%) (init = 1e-06)
    b:   0.10000000 +/- 0.02344604 (23.45%) (init = 0.5)
[[Correlations]] (unreported correlations are < 0.100)
    C(Di, b)  = +0.9643
    C(qi, Di) = +0.8828
    C(qi, b)  = +0.7547
30995.492770625617 0.0015632747769409377 0.10000000000006859


In [169]:
# harmonic
qi_har,Di_har=Harmonic_minimizer(t,q)

q_model_har= qi_har/(1+Di_har*t) 
# forecast rates
q_forecast_har = qi_har/(1+Di_har*t_forecast) 

[[Fit Statistics]]
    # fitting method   = least_squares
    # function evals   = 42
    # data points      = 2724
    # variables        = 2
    chi-square         = 1378.44771
    reduced chi-square = 0.50640989
    Akaike info crit   = -1851.43454
    Bayesian info crit = -1839.61483
[[Variables]]
    qi:  46253.7029 +/- 4414.67250 (9.54%) (init = 26049.78)
    Di:  0.00784455 +/- 8.6883e-04 (11.08%) (init = 1e-06)
[[Correlations]] (unreported correlations are < 0.100)
    C(qi, Di) = +0.9853


In [170]:
# plots
trace_data =go.Scatter(x=df["DATEPRD"],y=q, mode = "markers", name= "markers" )

# exponential 
trace_model_exp =go.Scatter(x=df["DATEPRD"],y=q_model_exp, mode = "lines", name= "mylines" )
trace_forecast_exp =go.Scatter(x=forecast_dates, y=q_forecast_exp, mode = "lines", name= "mylines" )

# hyperbolic
trace_model_hyper =go.Scatter(x=df["DATEPRD"],y=q_model_hyper, mode = "lines", name= "mylines" )
trace_forecast_hyper =go.Scatter(x=forecast_dates, y=q_forecast_hyper, mode = "lines", name= "mylines" )
    
# Harmonic
trace_model_har =go.Scatter(x=df["DATEPRD"],y=q_model_har, mode = "lines", name= "mylines" )
trace_forecast_har =go.Scatter(x=forecast_dates, y=q_forecast_har, mode = "lines", name= "mylines" )


data = [trace_data,trace_model_exp,trace_model_hyper,trace_model_har]

layout =go.Layout(title ="Line Charts")

fig =go.Figure(data=data, layout=layout)
pyo.plot(fig)

'temp-plot.html'